# Construct End-User Parquet Files

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import glob
import awkward as ak
import itertools
import yaml
import os
import sys
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
atl.ATLAS = "ColliderML"

sys.path.append("../")
from utils import load_root_file, load_hepmc_event
from edm4hep_utils import build_calo_df, build_particle_df, build_tracker_df, load_edm4hep_file
from clustering_metrics import evaluate_clustering, plot_clustering_metrics

from edm4hep_utils import pixel_readouts, strip_readouts
all_tracker_readouts = pixel_readouts + strip_readouts

## Roadmap

1. Load in edm4hep file
2. Load in ambi tracks csv 
3. Load in particles root
4. Load in measurements root
5. Load in trackstates ambi root
6. Load in tracksummary ambi root
7. Create track parquet object
8. Track object needs:
    - hits in the track (hit ID)
    - track fit parameters
    - track quality parameters
    - track states at IP, end-of-track, first hit, last hit

# Loading

## 1. Load in edm4hep file

In [2]:
edm4hep_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/edm4hep.root"
# edm4hep_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/chichi/v1/runs/0/edm4hep.root"
event = load_edm4hep_file(edm4hep_file, event_num=0)

In [3]:
event.keys()

dict_keys(['tracker_df', 'calo_hits_df', 'calo_contrib_df', 'particles_df', 'parents_df', 'daughters_df'])

In [174]:
def get_particle_ids_from_events(events, tracker_readouts):
    """Get particle IDs from events for each tracker readout.
    
    Args:
        events: uproot events object
        tracker_readouts: list of tracker readout names
        
    Returns:
        DataFrame containing event_id and particle_id columns
    """
    all_particle_ids = []
    for det in tracker_readouts:
        if det not in events:
            continue
        hits = events[det].arrays()
        hits_df = ak.to_dataframe(hits[[f"{det}.position.x", f"{det}.position.y", f"{det}.position.z"]]).reset_index(drop=False).rename(columns={"entry": "event_id", f"{det}.position.x": "x", f"{det}.position.y": "y", f"{det}.position.z": "z"}).drop(columns=["subentry"])
        particle_links = events[f"_{det}_MCParticle"].arrays()
        particle_links_df = ak.to_dataframe(particle_links[f"_{det}_MCParticle.index"]) \
            .reset_index(drop=False) \
            .rename(columns={"entry": "event_id", "values": "particle_id"}) \
            .drop(columns=["subentry"])
        # horizontal concat
        hits_df = pd.concat([hits_df, particle_links_df], axis=1)
        all_particle_ids.append(hits_df)

    particle_ids = pd.concat(all_particle_ids, ignore_index=True)
    return particle_ids

particle_ids = get_particle_ids_from_events(events, all_tracker_readouts)
particle_ids


,event_id,x,y,z,event_id,particle_id
0,0,-32.366956,1.492748,-401.924170,0,8352
1,0,2.450572,-31.891473,-346.345357,0,8350
2,0,41.731344,-165.855602,-459.294072,0,9944
3,0,42.007489,-165.670502,-459.724257,0,9944
4,0,-31.250231,6.906123,-266.890574,0,8347
...,...,...,...,...,...,...
174019,9,280.910245,-718.181977,1579.500000,9,503348
174020,9,112.264841,-755.496353,1874.500000,9,503348
174021,9,109.257986,-755.459817,1879.500000,9,503348
174022,9,955.142649,274.883962,2620.500000,9,503595


In [4]:
tracker_df = event["tracker_df"]
tracker_df.columns

Index(['cellID', 'EDep', 'time', 'pathLength', 'quality', 'x', 'y', 'z', 'px',
       'py', 'pz', 'particle_id', 'r', 'R', 'phi', 'theta', 'eta', 'pt',
       'detector'],
      dtype='object')

In [5]:
calo_df = event["calo_contrib_df"]
calo_df.columns

Index(['PDG', 'energy', 'time', 'x', 'y', 'z', 'particle_id', 'detector'], dtype='object')

In [11]:
parents_df = event["parents_df"]
parents_df.columns

particles_df = event["particles_df"]
# Create a column from the index
particles_df["particle_id"] = particles_df.index
particles_df.columns

Index(['PDG', 'generatorStatus', 'simulatorStatus', 'charge', 'time', 'mass',
       'vx', 'vy', 'vz', 'px', 'py', 'pz', 'endpoint_x', 'endpoint_y',
       'endpoint_z', 'parents_begin', 'parents_end', 'daughters_begin',
       'daughters_end', 'pt', 'p', 'eta', 'phi', 'particle_id'],
      dtype='object')

## 2. Load in ambi tracks csv 

In [13]:
tracks_csv_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/event000000000-tracks_ambi.csv"
tracks_df = pd.read_csv(tracks_csv_file)

In [14]:
tracks_df = tracks_df.sort_values(by="track_id").reset_index(drop=True)
tracks_df

,track_id,seed_id,particleId,nStates,nMajorityHits,nMeasurements,nOutliers,nHoles,nSharedHits,chi2,ndf,chi2/ndf,pT,eta,phi,truthMatchProbability,good/duplicate/fake,Hits_ID
0,0,26,1|3650|2|16|51835,26,14,14,0,0,0,19.24160,30,0.641387,1.16019,-0.109862,-2.93085,1.0,good,"[2901,2972,3313,4321,4341,5195,10453,11712,129..."
1,1,106,1|1482|18|11|9911,28,14,14,0,0,0,16.76540,29,0.578117,2.04052,1.620120,-3.08076,1.0,good,"[2915,3292,4314,5156,5180,10399,11683,12907,14..."
2,2,128,1|1172|18|6|12462,30,15,15,0,0,0,27.07430,31,0.873364,1.21133,1.605410,-2.96980,1.0,good,"[2913,2983,3321,4347,5201,10462,10434,11745,12..."
3,3,185,1|1172|18|6|12464,18,10,10,0,0,0,9.89781,26,0.380685,2.81650,1.132300,-2.81957,1.0,good,"[2981,3318,3342,4343,4360,5218,10457,11763,117..."
4,4,313,1|2112|16|13|44679,15,8,8,0,0,0,18.07020,22,0.821374,1.19249,0.584676,-2.91371,1.0,good,"[2928,3003,3323,4345,5209,5219,10458,11761,]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,105,4691,1|303|10|15|48253,29,17,17,0,0,0,23.86470,33,0.723172,1.19820,-0.182321,3.00169,1.0,good,"[2912,3257,3287,4294,5152,10420,10354,11694,11..."
106,106,4734,1|2802|18|7|18686,26,13,13,0,0,0,13.86090,30,0.462029,2.01138,1.226450,3.07505,1.0,good,"[2914,3290,4295,4312,5153,5160,10397,11679,129..."
107,107,4749,1|1973|18|9|33789,25,12,12,0,0,0,21.17900,31,0.683192,1.47404,-2.464820,2.99683,1.0,good,"[2808,2877,1566,1361,1262,1018,750,9414,9148,8..."
108,108,4799,1|4001|6|4|1478,24,11,11,0,0,0,24.39280,29,0.841130,1.18298,2.666500,3.01909,1.0,good,"[2871,3252,5637,5962,6567,6903,7237,14930,1530..."


## 3. Load in particles root

In [15]:
particles_root_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/particles.root"
particles_root_df = load_root_file(particles_root_file)
particles_root_df = particles_root_df[particles_root_df.event_id == 0]
particles_root_df

,event_id,particle_id,particle_type,process,vx,vy,vz,vt,px,py,...,vertex_primary,vertex_secondary,particle,generation,sub_particle,e_loss,total_x0,total_l0,number_of_hits,outcome
entry,,,,,,,,,,,,,,,,,,,,,
0,0,4504699155841029,-2,0,0.003543,-0.005813,-105.372108,2425.787842,-3.244334,1.407285,...,1,1,1,1,5,12.115084,0.0,0.0,0,0
1,0,4504699155841032,-1,0,0.003543,-0.005813,-105.372108,2425.787842,-2.774586,-0.292908,...,1,1,1,1,8,12.869830,0.0,0.0,0,0
2,0,4542082569470722,-211,0,0.003543,-0.005813,-105.372108,2425.787842,1.175650,0.272212,...,1,35,2,24,770,2.083743,0.0,0.0,0,0
3,0,4592660372048331,211,0,0.009781,0.014497,-116.807983,3995.353760,-0.969926,-0.885298,...,1,81,18,12,51659,5.592474,0.0,0.0,0,0
4,0,4621247405833537,3,0,0.003543,-0.005813,-105.372108,2425.787842,-1.653649,-0.264909,...,1,107,2,11,15681,1.554827,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
323,0,8988507592132378,-2212,0,0.003543,-0.005813,-105.372108,2425.787842,3.789078,-0.840354,...,1,4079,2,23,1818,37.361057,0.0,0.0,0,0
324,0,8988507592132379,213,0,0.003543,-0.005813,-105.372108,2425.787842,6.949781,-1.863585,...,1,4079,2,23,1819,68.278442,0.0,0.0,0,0
325,0,8989607103824618,11,0,0.643046,-0.055682,-106.382172,2427.019775,3.200787,-0.905479,...,1,4080,2,24,746,7.821862,0.0,0.0,0,0


In [16]:
particles_root_df.columns

Index(['event_id', 'particle_id', 'particle_type', 'process', 'vx', 'vy', 'vz',
       'vt', 'px', 'py', 'pz', 'm', 'q', 'eta', 'phi', 'pt', 'p',
       'vertex_primary', 'vertex_secondary', 'particle', 'generation',
       'sub_particle', 'e_loss', 'total_x0', 'total_l0', 'number_of_hits',
       'outcome'],
      dtype='object')

## 4. Load in measurements root

In [17]:
measurements_root_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/measurements.root"
measurements_root_df = load_root_file(measurements_root_file)
measurements_root_df = measurements_root_df[measurements_root_df.event_nr == 0]
measurements_root_df

,event_nr,volume_id,layer_id,surface_id,rec_loc0,rec_loc1,rec_time,var_loc0,var_loc1,var_time,...,true_y,true_z,true_incident_phi,true_incident_theta,residual_loc0,residual_loc1,residual_time,pull_loc0,pull_loc1,pull_time
entry,,,,,,,,,,,,,,,,,,,,,
0,0,16,4,1,-5.391152,13.295135,2195.870361,0.000225,0.000225,625.0,...,5.393176,-1515.599976,-1.459599,-1.473839,0.002024,-0.031604,19.788086,0.134913,-2.106921,0.791523
1,0,16,4,1,-4.674127,-12.982083,1841.585815,0.000225,0.000225,625.0,...,4.650855,-1515.599976,-1.572250,-1.527913,-0.023273,-0.007736,18.477905,-1.551501,-0.515747,0.739116
2,0,16,4,1,1.798209,14.153524,5433.405762,0.000225,0.000225,625.0,...,-1.796745,-1515.599976,-1.520368,-1.362961,0.001464,-0.000252,16.217285,0.097601,-0.016785,0.648691
3,0,16,4,1,-4.121328,-33.277374,5442.389160,0.000225,0.000225,625.0,...,4.109626,-1515.599976,-1.750785,-1.376686,-0.011702,0.020752,13.730469,-0.780137,1.383464,0.549219
4,0,16,4,1,8.683470,-26.376106,3865.373291,0.000225,0.000225,625.0,...,-8.630102,-1515.599976,-1.565840,-1.535524,0.053368,0.011538,28.286133,3.557841,0.769170,1.131445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22060,0,30,12,188,35.410561,NaN,NaN,0.005184,NaN,NaN,...,778.393188,3009.500000,1.726234,0.873742,-0.181332,NaN,NaN,-2.518495,NaN,NaN
22061,0,30,12,188,34.887432,NaN,NaN,0.005184,NaN,NaN,...,798.362793,3009.543213,0.879723,0.909441,-0.070179,NaN,NaN,-0.974708,NaN,NaN
22062,0,30,12,189,-14.325792,NaN,NaN,0.005184,NaN,NaN,...,826.348511,3025.500000,-0.974864,-2.480016,-0.074800,NaN,NaN,-1.038882,NaN,NaN


In [18]:
simhits_root_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/simhits.root"
simhits_root_df = load_root_file(simhits_root_file)
simhits_root_df = simhits_root_df[simhits_root_df.event_id == 0]
simhits_root_df

,event_id,geometry_id,particle_id,tx,ty,tz,tt,tpx,tpy,tpz,...,deltapx,deltapy,deltapz,deltae,index,volume_id,boundary_id,layer_id,approach_id,sensitive_id
entry,,,,,,,,,,,,,,,,,,,,,
0,0,1152921779484754177,5874690761757029,89.326736,5.393176,-1515.599976,2176.082275,0.249235,-0.286123,-2.562505,...,0.0,0.0,0.0,-0.000048,-1,16,0,4,0,1
1,0,1152921779484754177,6139673098346907,63.025654,4.650855,-1515.599976,1823.107910,0.516405,0.017491,-12.034775,...,0.0,0.0,0.0,-0.000034,-1,16,0,4,0,1
2,0,1152921779484754177,5765839279405280,90.153778,-1.796745,-1515.599976,5417.188477,0.013453,-0.003220,-0.063792,...,0.0,0.0,0.0,-0.000039,-1,16,0,4,0,1
3,0,1152921779484754177,5917571883897299,42.701874,4.109626,-1515.599976,5428.658691,0.008908,0.008245,-0.045315,...,0.0,0.0,0.0,-0.000047,-1,16,0,4,0,1
4,0,1152921779484754177,5348024591875968,49.612358,-8.630102,-1515.599976,3837.087158,0.394229,-0.055370,-11.171949,...,0.0,0.0,0.0,-0.000031,-1,16,0,4,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22060,0,2161728645771607042,8172670165793608,641.695312,778.393188,3009.500000,1461.297974,0.000915,0.001897,0.002473,...,0.0,0.0,0.0,-0.000085,-1,30,0,12,0,188
22061,0,2161728645771607042,6167161023350923,655.605896,798.362793,3009.543213,8740.767578,0.000030,0.000004,0.000027,...,0.0,0.0,0.0,-0.000002,-1,30,0,12,0,188
22062,0,2161728645771607298,7068760557851021,494.181335,826.348511,3025.500000,7913.167480,-0.032335,-0.021248,0.026641,...,0.0,0.0,0.0,-0.000980,-1,30,0,12,0,189


## 5. Load in trackstates ambi root

In [19]:
trackstates_root_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/trackstates_ambi.root"
trackstates_root_df = load_root_file(trackstates_root_file)
trackstates_root_df = trackstates_root_df[trackstates_root_df.event_nr == 0]
trackstates_root_df

,event_nr,track_nr,nStates,nMeasurements,nPredicted,nFiltered,nSmoothed,nUnbiased
entry,,,,,,,,
0,0,0,26,14,26,26,26,14
1,0,1,28,14,28,28,28,14
2,0,2,30,15,30,30,30,15
3,0,3,18,10,18,18,18,10
4,0,4,15,8,15,15,14,7
...,...,...,...,...,...,...,...,...
105,0,105,29,17,29,29,29,17
106,0,106,26,13,26,26,26,13
107,0,107,25,12,25,25,25,12


In [20]:
trackstates_root = uproot.open(trackstates_root_file)["trackstates"]
trackstates_root.arrays()[0]

<Record {event_nr: 0, track_nr: 0, ...} type='{event_nr: uint32, track_nr: ...'>

## 6. Load in tracksummary ambi root

In [24]:
tracksummary_root_file = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/tracksummary_ambi.root"
tracksummary_root_df = load_root_file(tracksummary_root_file)
tracksummary_root_df

,event_nr
entry,
0,0
1,1
2,2
3,3
4,4
5,5
6,6
7,7
8,8


In [25]:
tracksummary_root = uproot.open(tracksummary_root_file)["tracksummary"]

In [61]:
# Get the first event (index 0)
arrays = tracksummary_root.arrays()[0]

# Initialize empty dict
track_data = {}

# Loop through all fields except event_nr
for field in arrays.fields:
    if field == 'event_nr':
        continue
    try:
        array = ak.to_numpy(arrays[field])
        # Check that the field is one-dimensional
        assert len(array.shape) == 1, f"Field {field} is not one-dimensional"
        # Convert to numpy array
        track_data[field] = array
    except Exception as e:
        print(f"Skipping field {field} due to error: {e}")
        # These are likely variable-length arrays that need special handling
        continue

# Create DataFrame
track_fitting_df = pd.DataFrame(track_data).rename(columns={"track_nr": "track_id"})

# Print shape and columns to verify
print("DataFrame shape:", track_fitting_df.shape)
print("\nColumns:", track_fitting_df.columns.tolist())
print("\nFirst few rows:")
print(track_fitting_df.head())

Skipping field measurementChi2 due to error: cannot convert to RegularArray because subarray lengths are not regular (in compiled code: https://github.com/scikit-hep/awkward/blob/awkward-cpp-44/awkward-cpp/src/cpu-kernels/awkward_ListOffsetArray_toRegularArray.cpp#L22)
Skipping field outlierChi2 due to error: cannot convert to RegularArray because subarray lengths are not regular (in compiled code: https://github.com/scikit-hep/awkward/blob/awkward-cpp-44/awkward-cpp/src/cpu-kernels/awkward_ListOffsetArray_toRegularArray.cpp#L22)
Skipping field measurementVolume due to error: cannot convert to RegularArray because subarray lengths are not regular (in compiled code: https://github.com/scikit-hep/awkward/blob/awkward-cpp-44/awkward-cpp/src/cpu-kernels/awkward_ListOffsetArray_toRegularArray.cpp#L22)
Skipping field measurementLayer due to error: cannot convert to RegularArray because subarray lengths are not regular (in compiled code: https://github.com/scikit-hep/awkward/blob/awkward-cpp-

In [27]:
track_fitting_df

,track_id,nStates,nMeasurements,nOutliers,nHoles,nSharedHits,chi2Sum,NDF,nMajorityHits,majorityParticleId,...,cov_eQOP_ePHI,cov_eQOP_eTHETA,cov_eQOP_eQOP,cov_eQOP_eT,cov_eT_eLOC0,cov_eT_eLOC1,cov_eT_ePHI,cov_eT_eTHETA,cov_eT_eQOP,cov_eT_eT
0,0,16,8,0,0,0,18.617577,21,8,4814761452736345,...,5.431699e-06,1.638425e-08,1.439984e-04,-2.155326e-04,0.000197,-0.001331,-8.112069e-06,-2.398773e-05,-2.155326e-04,124.827492
1,1,28,16,0,0,0,31.690992,34,4294967295,18446744073709551615,...,1.408219e-06,3.447064e-09,3.116564e-05,5.084811e-05,-0.000086,-0.002322,3.654124e-06,-2.995609e-05,5.084811e-05,104.047882
2,2,25,12,0,1,0,16.545481,25,12,4837851196919643,...,6.667299e-07,5.823173e-09,1.493227e-05,-1.223941e-05,0.000020,-0.000703,-6.018636e-07,-1.033063e-05,-1.223941e-05,124.826828
3,3,26,14,0,0,0,14.629357,29,14,4843348755131606,...,1.689954e-06,2.610214e-09,4.618774e-05,6.481825e-05,-0.000031,-0.001418,2.137611e-06,-2.745283e-05,6.481825e-05,124.827309
4,4,26,14,0,0,0,15.299042,30,14,4623446429431230,...,8.645520e-08,2.057700e-09,1.800346e-06,-6.360641e-07,0.000008,-0.000107,-5.570396e-08,-1.688303e-06,-6.360641e-07,104.046120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,106,24,12,0,0,0,13.410120,24,12,4623446429431217,...,5.043538e-08,-4.987204e-10,5.390786e-07,-8.500898e-08,0.000011,-0.000090,-4.203214e-08,-1.006298e-06,-8.500898e-08,155.978867
107,107,27,14,0,1,0,18.597334,27,14,4674023964381000,...,1.056928e-07,1.629963e-09,1.973583e-06,-7.364654e-07,0.000013,-0.000155,-8.410936e-08,-2.364179e-06,-7.364654e-07,155.978882
108,108,30,18,0,0,0,25.300423,33,18,5984641992485067,...,6.496944e-07,1.314954e-09,1.861456e-05,-2.014343e-05,0.000019,-0.000016,-5.493535e-07,-3.922406e-07,-2.014343e-05,124.826416
109,109,30,18,0,0,0,41.443546,35,18,5984641992485066,...,1.777637e-06,-1.468426e-09,5.680400e-05,1.013819e-04,-0.000058,0.000582,2.952582e-06,1.490701e-05,1.013819e-04,104.046448


# Track Object Construction

## Create track hdf5 object

In [50]:
import h5py
import numpy as np
from typing import Dict, Any

def convert_hit_ids(hit_ids_str: str) -> np.ndarray:
    """
    Convert string representation of hit IDs '[1,2,3,...]' to numpy array.
    
    Args:
        hit_ids_str: String representation of hit IDs array
        
    Returns:
        numpy array of integers
    """
    # Remove brackets and split by comma
    hit_ids = hit_ids_str.strip('[]').split(',')
    # Convert to integers
    return np.array([int(x) for x in hit_ids if x.strip()], dtype=np.int32)

def build_hdf5_tracks(
    df: pd.DataFrame,
    output_file: str
) -> None:
    """
    Build HDF5 file with event/track/hit hierarchy using variable-length arrays.
    
    Structure:
    /events/
        /event_0/
            /tracks      # Dataset containing track properties
            /hit_ids    # Dataset containing variable-length hit arrays per track
        /event_1/
            ...
    
    Args:
        df: DataFrame containing track data with hit_ids column
        output_file: Path to output HDF5 file
    """
    with h5py.File(output_file, 'a') as f:
        # Create events group if it doesn't exist
        if 'events' not in f:
            events_group = f.create_group('events')
        else:
            events_group = f['events']
            
        # Group DataFrame by event_id
        for event_id, event_df in df.groupby('event_id'):
            # Create event group
            event_group = events_group.create_group(f'event_{event_id}')
            
            # Store track data (all columns except hit_ids and event_id)
            track_data = event_df.drop(columns=['hit_ids', 'event_id'])
            event_group.create_dataset('tracks', data=track_data.to_records(index=False))
            
            # Store hit arrays using variable-length array
            hit_arrays = event_df['hit_ids'].values
            event_group.create_dataset(
                'hit_ids',
                data=hit_arrays,
                dtype=h5py.vlen_dtype(np.dtype('int32')),
                compression="gzip",
                compression_opts=9
            )

def read_event_tracks(
    input_file: str,
    event_id: int
) -> Dict[str, Any]:
    """
    Read track data for a specific event.
    
    Args:
        input_file: Path to HDF5 file
        event_id: Event ID to read
        
    Returns:
        Dictionary containing:
            - tracks: Structured array of track properties
            - hit_ids: List of hit ID arrays for each track
    """
    with h5py.File(input_file, 'r') as f:
        event_group = f[f'events/event_{event_id}']
        
        return {
            'tracks': event_group['tracks'][:],
            'hit_ids': event_group['hit_ids'][:]
        }

## Map EDM4Hep Particle IDs and ACTS Particle Barcodes

In [28]:
def analyze_coordinate_matches(comparison_df, tolerance=1e-3):
    """
    Analyze how well coordinates match between two sets of points in comparison_df.
    
    Args:
        comparison_df: DataFrame containing x,y,z and tx,ty,tz columns to compare
        tolerance: Absolute tolerance for np.isclose comparison
        
    Returns:
        dict: Dictionary containing match statistics and example mismatches
    """
    # Check if values are close
    is_close = np.isclose(
        comparison_df[["x", "y", "z"]],
        comparison_df[["tx", "ty", "tz"]],
        atol=tolerance
    )

    # Calculate matches per coordinate
    x_matches = (is_close[:, 0].sum() / len(comparison_df)) * 100
    y_matches = (is_close[:, 1].sum() / len(comparison_df)) * 100
    z_matches = (is_close[:, 2].sum() / len(comparison_df)) * 100

    total_matches = np.mean([x_matches, y_matches, z_matches])
    assert total_matches > 99.9, f"Simhits to edm4hep hit match below threshold. Matching percentage: {total_matches:.2f}%"

In [153]:
def create_particle_barcode_map(tracker_df, simhits_root_df):
    """
    Create a mapping between particle barcodes and particle IDs by matching hit coordinates.
    
    Args:
        tracker_df: DataFrame containing edm4hep hits with x,y,z coordinates
        simhits_root_df: DataFrame containing ROOT simhits with tx,ty,tz coordinates
        
    Returns:
        DataFrame: Mapping between particle_barcode and particle_id
    """
    # First set dtype of x,y,z and tx,ty,tz to float32
    tracker_df[["x", "y", "z"]] = tracker_df[["x", "y", "z"]].astype(np.float32)
    simhits_root_df[["tx", "ty", "tz"]] = simhits_root_df[["tx", "ty", "tz"]].astype(np.float32)

    # Reset indices and sort both DataFrames
    tracker_df_sorted = tracker_df.reset_index(drop=True).sort_values(by=["x", "y", "z"])
    simhits_df_sorted = simhits_root_df.reset_index(drop=True).sort_values(by=["tx", "ty", "tz"])

    # Rename particle_id to particle_barcode
    tracker_df_sorted = tracker_df_sorted.rename(columns={"particle_id": "particle_barcode"})

    # Now concatenate
    comparison_df = pd.concat(
        [
            tracker_df_sorted[["x", "y", "z", "particle_barcode"]].reset_index(drop=True),
            simhits_df_sorted[["tx", "ty", "tz", "particle_id"]].reset_index(drop=True)
        ],
        axis=1
    )

    analyze_coordinate_matches(comparison_df)

    particle_barcode_map = comparison_df[["particle_barcode", "particle_id"]].drop_duplicates()
    particle_barcode_map = particle_barcode_map.set_index('particle_id')['particle_barcode']
    return particle_barcode_map

particle_barcode_map = create_particle_barcode_map(tracker_df, simhits_root_df)
particle_barcode_map

particle_id
5845003847824473    284097
7260075580901193     97309
8995104811581868      3358
7554744730876962     66624
8129789278719944     94182
                     ...  
8200157754724050    291493
8992905638842669    404466
4836751819432257    219993
7362330195852973     16444
8645460232209618    119919
Name: particle_barcode, Length: 3869, dtype: int32

In [102]:
tracks_df

,track_id,seed_id,particleId,nStates,nMajorityHits,nMeasurements,nOutliers,nHoles,nSharedHits,chi2,ndf,chi2/ndf,pT,eta,phi,truthMatchProbability,good/duplicate/fake,Hits_ID
0,0,26,1|3650|2|16|51835,26,14,14,0,0,0,19.24160,30,0.641387,1.16019,-0.109862,-2.93085,1.0,good,"[2901,2972,3313,4321,4341,5195,10453,11712,129..."
1,1,106,1|1482|18|11|9911,28,14,14,0,0,0,16.76540,29,0.578117,2.04052,1.620120,-3.08076,1.0,good,"[2915,3292,4314,5156,5180,10399,11683,12907,14..."
2,2,128,1|1172|18|6|12462,30,15,15,0,0,0,27.07430,31,0.873364,1.21133,1.605410,-2.96980,1.0,good,"[2913,2983,3321,4347,5201,10462,10434,11745,12..."
3,3,185,1|1172|18|6|12464,18,10,10,0,0,0,9.89781,26,0.380685,2.81650,1.132300,-2.81957,1.0,good,"[2981,3318,3342,4343,4360,5218,10457,11763,117..."
4,4,313,1|2112|16|13|44679,15,8,8,0,0,0,18.07020,22,0.821374,1.19249,0.584676,-2.91371,1.0,good,"[2928,3003,3323,4345,5209,5219,10458,11761,]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,105,4691,1|303|10|15|48253,29,17,17,0,0,0,23.86470,33,0.723172,1.19820,-0.182321,3.00169,1.0,good,"[2912,3257,3287,4294,5152,10420,10354,11694,11..."
106,106,4734,1|2802|18|7|18686,26,13,13,0,0,0,13.86090,30,0.462029,2.01138,1.226450,3.07505,1.0,good,"[2914,3290,4295,4312,5153,5160,10397,11679,129..."
107,107,4749,1|1973|18|9|33789,25,12,12,0,0,0,21.17900,31,0.683192,1.47404,-2.464820,2.99683,1.0,good,"[2808,2877,1566,1361,1262,1018,750,9414,9148,8..."
108,108,4799,1|4001|6|4|1478,24,11,11,0,0,0,24.39280,29,0.841130,1.18298,2.666500,3.01909,1.0,good,"[2871,3252,5637,5962,6567,6903,7237,14930,1530..."


In [103]:
track_fitting_df

,track_id,nStates,nMeasurements,nOutliers,nHoles,nSharedHits,chi2Sum,NDF,nMajorityHits,majorityParticleId,...,cov_eQOP_ePHI,cov_eQOP_eTHETA,cov_eQOP_eQOP,cov_eQOP_eT,cov_eT_eLOC0,cov_eT_eLOC1,cov_eT_ePHI,cov_eT_eTHETA,cov_eT_eQOP,cov_eT_eT
0,0,26,14,0,0,0,19.241606,30,14,8516817103407739,...,1.556036e-06,1.222283e-09,0.000053,-7.981353e-05,0.000048,-0.000175,-0.000002,-0.000005,-7.981353e-05,104.046219
1,1,28,14,0,0,0,16.765385,29,14,6133076162455223,...,2.959903e-07,-1.268731e-09,0.000003,-3.834591e-06,0.000112,0.005312,-0.000002,0.000023,-3.834591e-06,124.831306
2,2,30,15,0,0,0,27.074282,31,15,5792227557519534,...,9.129250e-07,1.892423e-09,0.000012,1.720043e-05,-0.000105,0.014083,0.000003,0.000066,1.720043e-05,124.839417
3,3,18,10,0,0,0,9.897814,26,10,5792227557519536,...,9.484326e-07,-1.784887e-08,0.000016,-1.381000e-05,0.000146,0.001426,-0.000003,0.000012,-1.381000e-05,104.047249
4,4,15,8,0,0,0,18.070225,22,8,6825768454565511,...,8.743967e-06,-9.486366e-09,0.000247,4.126023e-04,-0.000320,0.001106,0.000014,0.000023,4.126023e-04,104.047356
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,105,29,17,0,0,0,23.864670,33,17,4836751819390077,...,1.618563e-06,6.248948e-09,0.000050,7.273889e-05,-0.000027,-0.000294,0.000002,-0.000007,7.273889e-05,124.826553
106,106,26,13,0,0,0,13.860875,30,13,7584431510866174,...,7.671558e-07,-1.109824e-08,0.000012,1.254951e-05,0.000143,0.002720,-0.000002,0.000022,1.254951e-05,104.048386
107,107,25,12,0,0,0,21.178953,31,12,6672936371586045,...,5.801658e-07,-3.234888e-09,0.000003,1.159434e-08,-0.001003,-0.104891,0.000009,-0.000097,1.159434e-08,89.300438
108,108,24,11,0,0,0,24.392767,29,11,8902745751029190,...,1.358523e-06,-2.901632e-09,0.000007,-1.425466e-05,0.001933,0.305671,-0.000044,0.000186,-1.425466e-05,89.499939


In [104]:
tracks_df.Hits_ID.apply(convert_hit_ids)[0]

array([ 2901,  2972,  3313,  4321,  4341,  5195, 10453, 11712, 12938,
       13789, 18592, 18593, 19806, 19811], dtype=int32)

In [109]:
majority_particle_ids

Index([  946,  6025,  5728,  5730,  4432,  6031,  5731,  6124,  6158,  1076,
       ...
        6109, 28624,  6235,  2109,   952,  3172,  6117,  6436,  2248,   906],
      dtype='int32', length=110)

In [112]:
particles_df.iloc[946]

PDG                2.110000e+02
generatorStatus    1.000000e+00
simulatorStatus    8.388608e+07
charge             1.000000e+00
time               8.091558e+00
mass               1.395700e-01
vx                 3.542614e-03
vy                -5.812524e-03
vz                -1.053721e+02
px                -1.137666e+00
py                -2.454168e-01
pz                -1.278222e-01
endpoint_x        -1.323035e+03
endpoint_y         1.822930e+02
endpoint_z        -2.513340e+02
parents_begin      3.791000e+03
parents_end        3.829000e+03
daughters_begin    5.002000e+03
daughters_end      5.003000e+03
pt                 1.163836e+00
p                  1.170834e+00
eta               -1.096088e-01
phi               -2.929129e+00
Name: 946, dtype: float64

In [107]:
particle_barcode_map[8516817103407739]

np.int32(946)

In [97]:
track_hits

,event_id,geometry_id,particle_id,tx,ty,tz,tt,tpx,tpy,tpz,...,deltapx,deltapy,deltapz,deltae,index,volume_id,boundary_id,layer_id,approach_id,sensitive_id
entry,,,,,,,,,,,,,,,,,,,,,
2901,0,1224979236083781120,8516817103407739,-31.402458,-6.506118,-108.896530,2458.344971,-1.141444,-0.225955,-0.128260,...,0.0,0.0,0.0,-0.000029,-1,17,0,2,0,206
2972,0,1224979236083784704,8516817103407739,-32.847523,-6.791385,-109.059204,2459.840576,-1.141468,-0.225145,-0.128698,...,0.0,0.0,0.0,-0.000032,-1,17,0,2,0,220
3313,0,1224979373522716672,8516817103407739,-66.929733,-13.140807,-112.888313,2494.967041,-1.139637,-0.201245,-0.127730,...,0.0,0.0,0.0,-0.000047,-1,17,0,4,0,136
4321,0,1224979510961709568,8516817103407739,-111.696671,-20.513559,-117.939621,2540.945312,-1.143090,-0.174268,-0.128925,...,0.0,0.0,0.0,-0.000035,-1,17,0,6,0,290
4341,0,1224979510961713152,8516817103407739,-113.351379,-20.764534,-118.124908,2542.642578,-1.143333,-0.172890,-0.127841,...,0.0,0.0,0.0,-0.000047,-1,17,0,6,0,304
5195,0,1224979648400720384,8516817103407739,-167.730179,-28.175749,-124.234085,2598.260986,-1.146908,-0.139557,-0.128177,...,0.0,0.0,0.0,-0.000041,-1,17,0,8,0,514
10453,0,1729382394349304576,8516817103407739,-256.825806,-36.880558,-134.158722,2689.022705,-1.150339,-0.084795,-0.128512,...,0.0,0.0,0.0,-0.000059,-1,24,0,2,0,315
11712,0,1729382531788301056,8516817103407739,-357.789062,-41.699539,-145.363358,2791.463867,-1.152230,-0.025328,-0.128172,...,0.0,0.0,0.0,-0.000087,-1,24,0,4,0,483
12938,0,1729382669227324416,8516817103407739,-498.337646,-39.643845,-160.995667,2933.945801,-1.150329,0.060066,-0.126475,...,0.0,0.0,0.0,-0.000064,-1,24,0,6,0,756


In [100]:
particle_barcode_map

,particle_barcode,particle_id
0,284097,5845003847824473
1,97309,7260075580901193
2,3358,8995104811581868
4,66624,7554744730876962
5,94182,8129789278719944
...,...,...
22045,291493,8200157754724050
22048,404466,8992905638842669
22049,219993,4836751819432257
22050,16444,7362330195852973


In [66]:
mapping

particle_id
5845003847824473    284097
7260075580901193     97309
8995104811581868      3358
7554744730876962     66624
8129789278719944     94182
                     ...  
8200157754724050    291493
8992905638842669    404466
4836751819432257    219993
7362330195852973     16444
8645460232209618    119919
Name: particle_barcode, Length: 3869, dtype: int32

In [70]:
# Map the particle Ids in the track fitting df to the particle barcodes
track_fitting_df["majorityParticleId"][track_fitting_df["majorityParticleId"].map(mapping).isna()]

55     18446744073709551615
101    18446744073709551615
Name: majorityParticleId, dtype: uint64

In [71]:
track_fitting_df[track_fitting_df["majorityParticleId"] == 18446744073709551615]

,track_id,nStates,nMeasurements,nOutliers,nHoles,nSharedHits,chi2Sum,NDF,nMajorityHits,majorityParticleId,...,cov_eQOP_ePHI,cov_eQOP_eTHETA,cov_eQOP_eQOP,cov_eQOP_eT,cov_eT_eLOC0,cov_eT_eLOC1,cov_eT_ePHI,cov_eT_eTHETA,cov_eT_eQOP,cov_eT_eT
55,55,23,11,0,0,0,18.627665,30,4294967295,18446744073709551615,...,2.943617e-07,-7.781583e-11,6.677895e-07,-7.940134e-07,0.000052,0.024217,-8.106700e-07,0.000009,-7.940134e-07,78.081177
101,101,25,11,0,0,0,29.477345,28,4294967295,18446744073709551615,...,1.302450e-06,-4.940433e-11,7.370985e-06,-9.978618e-06,-0.000153,-0.350132,-4.528409e-06,-0.000269,-9.978618e-06,104.392120


In [89]:
track_fitting_df

,track_id,nStates,nMeasurements,nOutliers,nHoles,nSharedHits,chi2Sum,NDF,nMajorityHits,majorityParticleId,...,cov_eQOP_ePHI,cov_eQOP_eTHETA,cov_eQOP_eQOP,cov_eQOP_eT,cov_eT_eLOC0,cov_eT_eLOC1,cov_eT_ePHI,cov_eT_eTHETA,cov_eT_eQOP,cov_eT_eT
0,0,26,14,0,0,0,19.241606,30,14,8516817103407739,...,1.556036e-06,1.222283e-09,0.000053,-7.981353e-05,0.000048,-0.000175,-0.000002,-0.000005,-7.981353e-05,104.046219
1,1,28,14,0,0,0,16.765385,29,14,6133076162455223,...,2.959903e-07,-1.268731e-09,0.000003,-3.834591e-06,0.000112,0.005312,-0.000002,0.000023,-3.834591e-06,124.831306
2,2,30,15,0,0,0,27.074282,31,15,5792227557519534,...,9.129250e-07,1.892423e-09,0.000012,1.720043e-05,-0.000105,0.014083,0.000003,0.000066,1.720043e-05,124.839417
3,3,18,10,0,0,0,9.897814,26,10,5792227557519536,...,9.484326e-07,-1.784887e-08,0.000016,-1.381000e-05,0.000146,0.001426,-0.000003,0.000012,-1.381000e-05,104.047249
4,4,15,8,0,0,0,18.070225,22,8,6825768454565511,...,8.743967e-06,-9.486366e-09,0.000247,4.126023e-04,-0.000320,0.001106,0.000014,0.000023,4.126023e-04,104.047356
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,105,29,17,0,0,0,23.864670,33,17,4836751819390077,...,1.618563e-06,6.248948e-09,0.000050,7.273889e-05,-0.000027,-0.000294,0.000002,-0.000007,7.273889e-05,124.826553
106,106,26,13,0,0,0,13.860875,30,13,7584431510866174,...,7.671558e-07,-1.109824e-08,0.000012,1.254951e-05,0.000143,0.002720,-0.000002,0.000022,1.254951e-05,104.048386
107,107,25,12,0,0,0,21.178953,31,12,6672936371586045,...,5.801658e-07,-3.234888e-09,0.000003,1.159434e-08,-0.001003,-0.104891,0.000009,-0.000097,1.159434e-08,89.300438
108,108,24,11,0,0,0,24.392767,29,11,8902745751029190,...,1.358523e-06,-2.901632e-09,0.000007,-1.425466e-05,0.001933,0.305671,-0.000044,0.000186,-1.425466e-05,89.499939


In [87]:
simhits_root_df.iloc[tracks_df.Hits_ID.apply(convert_hit_ids)[55]]

,event_id,geometry_id,particle_id,tx,ty,tz,tt,tpx,tpy,tpz,...,deltapx,deltapy,deltapz,deltae,index,volume_id,boundary_id,layer_id,approach_id,sensitive_id
entry,,,,,,,,,,,,,,,,,,,,,
2415,0,1224979236083757056,5611907383363589,30.724005,-8.787744,197.037659,2730.475830,8.087699,-2.306390,79.615845,...,0.0,0.0,0.0,-0.000365,-1,17,0,2,0,112
5739,0,1297036830121660673,5611907383363589,73.282898,-20.891596,615.599976,3150.838135,7.877717,-2.227542,77.452316,...,0.0,0.0,0.0,-0.000059,-1,18,0,2,0,17
6071,0,1297036967560614145,5611907383363589,83.453255,-23.761940,715.599976,3251.395020,7.877826,-2.220783,77.451080,...,0.0,0.0,0.0,-0.000024,-1,18,0,4,0,17
6437,0,1297037104999567617,5611907383363589,95.661507,-27.194534,835.599976,3372.063232,7.880066,-2.211988,77.449989,...,0.0,0.0,0.0,-0.000036,-1,18,0,6,0,17
6983,0,1297037242438530306,5611907383363589,110.741699,-31.418447,983.799988,3521.088379,7.516060,-2.099638,73.839500,...,0.0,0.0,0.0,-0.000038,-1,18,0,8,0,53
7314,0,1297037379877483778,5611907383363589,124.995911,-35.389194,1123.800049,3661.868164,7.519362,-2.091219,73.838295,...,0.0,0.0,0.0,-0.000040,-1,18,0,10,0,53
7644,0,1297037517316437250,5611907383363589,145.369492,-41.037304,1323.800049,3862.982666,7.522121,-2.080290,73.837326,...,0.0,0.0,0.0,-0.000036,-1,18,0,12,0,53
7977,0,1297037654755390722,5611907383363589,165.749329,-46.655674,1523.800049,4064.096924,7.525348,-2.067231,73.835938,...,0.0,0.0,0.0,-0.000033,-1,18,0,14,0,53
15418,0,1801440400704021505,5611907383363589,234.466141,-65.289352,2197.500000,4741.611328,6.595011,-1.764441,64.594986,...,0.0,0.0,0.0,-0.000065,-1,25,0,8,0,36


In [88]:
simhits_root_df.iloc[tracks_df.Hits_ID.apply(convert_hit_ids)[101]]

,event_id,geometry_id,particle_id,tx,ty,tz,tt,tpx,tpy,tpz,...,deltapx,deltapy,deltapz,deltae,index,volume_id,boundary_id,layer_id,approach_id,sensitive_id
entry,,,,,,,,,,,,,,,,,,,,,
2825,0,1224979236083777280,7201801498029976,-30.160374,10.913317,-208.405472,-869.726685,-0.157199,0.057088,-1.075131,...,0.0,0.0,0.0,-0.000123,-1,17,0,2,0,191
3220,0,1224979373522703616,7201801498029977,-63.765057,23.346043,-438.093689,-637.040833,-1.010602,0.385428,-6.928097,...,0.0,0.0,0.0,-0.000247,-1,17,0,4,0,85
1569,0,1152922604118475777,7201801498029977,-89.831100,33.678192,-616.799988,-456.491974,-1.006414,0.409241,-6.922508,...,0.0,0.0,0.0,-0.000061,-1,16,0,16,0,4
1482,0,1152922466679529730,7201801498029977,-105.176880,39.973232,-722.599976,-349.399658,-0.973150,0.404414,-6.725176,...,0.0,0.0,0.0,-0.000063,-1,16,0,14,0,33
1256,0,1152922329240576258,7201801498029977,-122.491966,47.281364,-842.599976,-227.936813,-0.968385,0.413963,-6.722043,...,0.0,0.0,0.0,-0.000036,-1,16,0,12,0,33
1007,0,1152922191801622786,7201801498029977,-142.612610,56.018867,-982.599976,-86.228683,-0.961715,0.425459,-6.721207,...,0.0,0.0,0.0,-0.000036,-1,16,0,10,0,33
9403,0,1657325350067112705,7201801498029977,-221.841721,93.267426,-1542.500000,480.539032,-0.936242,0.467446,-6.696725,...,0.0,0.0,0.0,-0.000070,-1,23,0,10,0,11
9138,0,1657325212628159233,7201801498029977,-263.457214,114.882034,-1842.500000,784.182190,-0.922287,0.494495,-6.695625,...,0.0,0.0,0.0,-0.000064,-1,23,0,8,0,11
8837,0,1657325075189205761,7201801498029977,-311.280121,141.446136,-2192.500000,1138.432007,-0.906551,0.521765,-6.695396,...,0.0,0.0,0.0,-0.000100,-1,23,0,6,0,11


In [81]:
tracks_df.iloc[55]

track_id                                                                55
seed_id                                                               2083
particleId                                                1|1008|2|25|2053
nStates                                                                 23
nMajorityHits                                                           11
nMeasurements                                                           11
nOutliers                                                                0
nHoles                                                                   0
nSharedHits                                                              0
chi2                                                               18.6277
ndf                                                                     30
chi2/ndf                                                          0.620922
pT                                                                 7.08971
eta                      

In [82]:
tracks_df.iloc[101]

track_id                                                               101
seed_id                                                               4571
particleId                                               1|2454|20|8|28569
nStates                                                                 25
nMajorityHits                                                           10
nMeasurements                                                           11
nOutliers                                                                0
nHoles                                                                   0
nSharedHits                                                              0
chi2                                                               29.4773
ndf                                                                     28
chi2/ndf                                                           1.05276
pT                                                                 1.08512
eta                      

In [75]:
track_fitting_df.majorityParticleId.dtype

dtype('uint64')

In [76]:
simhits_root_df.particle_id.dtype

dtype('uint64')

In [74]:
simhits_root_df[simhits_root_df.particle_id == 18446744073709551615]

,event_id,geometry_id,particle_id,tx,ty,tz,tt,tpx,tpy,tpz,...,deltapx,deltapy,deltapz,deltae,index,volume_id,boundary_id,layer_id,approach_id,sensitive_id
entry,,,,,,,,,,,,,,,,,,,,,


In [42]:
track_fitting_df.columns

Index(['track_nr', 'nStates', 'nMeasurements', 'nOutliers', 'nHoles',
       'nSharedHits', 'chi2Sum', 'NDF', 'nMajorityHits', 'majorityParticleId',
       'trackClassification', 't_charge', 't_time', 't_vx', 't_vy', 't_vz',
       't_px', 't_py', 't_pz', 't_theta', 't_phi', 't_eta', 't_p', 't_pT',
       't_d0', 't_z0', 'hasFittedParams', 'eLOC0_fit', 'eLOC1_fit', 'ePHI_fit',
       'eTHETA_fit', 'eQOP_fit', 'eT_fit', 'err_eLOC0_fit', 'err_eLOC1_fit',
       'err_ePHI_fit', 'err_eTHETA_fit', 'err_eQOP_fit', 'err_eT_fit',
       'res_eLOC0_fit', 'res_eLOC1_fit', 'res_ePHI_fit', 'res_eTHETA_fit',
       'res_eQOP_fit', 'res_eT_fit', 'pull_eLOC0_fit', 'pull_eLOC1_fit',
       'pull_ePHI_fit', 'pull_eTHETA_fit', 'pull_eQOP_fit', 'pull_eT_fit',
       'cov_eLOC0_eLOC0', 'cov_eLOC0_eLOC1', 'cov_eLOC0_ePHI',
       'cov_eLOC0_eTHETA', 'cov_eLOC0_eQOP', 'cov_eLOC0_eT', 'cov_eLOC1_eLOC0',
       'cov_eLOC1_eLOC1', 'cov_eLOC1_ePHI', 'cov_eLOC1_eTHETA',
       'cov_eLOC1_eQOP', 'cov_eLOC1_eT', '

In [53]:
track_fitting_df.eT_fit - track_fitting_df.t_time

0      -2.783936
1     -12.269775
2       7.022217
3     -13.249512
4      -7.668564
         ...    
105    -6.257111
106    19.145020
107     7.693115
108     7.146973
109     9.755859
Length: 110, dtype: float32

In [54]:
track_fitting_df.res_eT_fit

0      -2.783936
1     -12.269775
2       7.022217
3     -13.249512
4      -7.668564
         ...    
105    -6.257111
106    19.145020
107     7.693115
108     7.146973
109     9.755859
Name: res_eT_fit, Length: 110, dtype: float32

In [51]:
track_fitting_df.eLOC1_fit - track_fitting_df.t_z0

0     -0.000664
1      0.004898
2     -0.148483
3      0.014557
4      0.046921
         ...   
105   -0.016987
106    0.045769
107   -0.659187
108    0.305862
109   -0.002571
Length: 110, dtype: float32

In [52]:
track_fitting_df.res_eLOC1_fit

0     -0.000664
1      0.004898
2     -0.148483
3      0.014557
4      0.046921
         ...   
105   -0.016987
106    0.045769
107   -0.659187
108    0.305862
109   -0.002571
Name: res_eLOC1_fit, Length: 110, dtype: float32

In [116]:
track_finding_data = {
    "event_id": 0,
    "track_id": tracks_df.track_id.values,
    "num_hits": tracks_df.nMeasurements.values,
    "num_outliers": tracks_df.nOutliers.values,
    "num_holes": tracks_df.nHoles.values,
    "num_shared_hits": tracks_df.nSharedHits.values,
    "chi2": tracks_df.chi2.values,
    "hit_ids": tracks_df.Hits_ID.apply(convert_hit_ids).values,
    "majority_particle_id": majority_particle_ids.values,
}

track_fitting_data = {
    "event_id": 0,
    "track_id": track_fitting_df.track_id.values,
    "d0": track_fitting_df.eLOC0_fit.values,
    "z0": track_fitting_df.eLOC1_fit.values,
    "phi": track_fitting_df.ePHI_fit.values,
    "theta": track_fitting_df.eTHETA_fit.values,
    "qop": track_fitting_df.eQOP_fit.values,
    "time": track_fitting_df.eT_fit.values,
 }

full_track_df = pd.DataFrame(track_finding_data)
full_track_df = full_track_df.merge(pd.DataFrame(track_fitting_data), on=["event_id", "track_id"])
full_track_df

,event_id,track_id,num_hits,num_outliers,num_holes,num_shared_hits,chi2,hit_ids,majority_particle_id,d0,z0,phi,theta,qop,time
0,0,0,14,0,0,0,19.24160,"[2901, 2972, 3313, 4321, 4341, 5195, 10453, 11...",946,0.062087,-105.373024,-2.930852,1.680438,0.856753,2423.003906
1,0,1,14,0,0,0,16.76540,"[2915, 3292, 4314, 5156, 5180, 10399, 11683, 1...",6025,-0.092893,-116.777306,-3.080758,0.390704,0.186638,3983.083740
2,0,2,15,0,0,0,27.07430,"[2913, 2983, 3321, 4347, 5201, 10462, 10434, 1...",5728,-0.045445,-116.927620,-2.969800,0.396343,-0.318697,4002.375732
3,0,3,10,0,0,0,9.89781,"[2981, 3318, 3342, 4343, 4360, 5218, 10457, 11...",5730,0.049636,-116.774132,-2.819574,0.623558,0.207323,3982.104004
4,0,4,8,0,0,0,18.07020,"[2928, 3003, 3323, 4345, 5209, 5219, 10458, 11...",4432,0.001409,72.367752,-2.913705,1.016841,-0.713171,-183.507492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,0,105,17,0,0,0,23.86470,"[2912, 3257, 3287, 4294, 5152, 10420, 10354, 1...",3172,-0.020431,-41.542458,3.001691,1.752115,-0.820906,341.259796
106,0,106,13,0,0,0,13.86090,"[2914, 3290, 4295, 4312, 5153, 5160, 10397, 11...",6117,-0.044598,-116.748528,3.075047,0.570654,-0.268563,4014.498535
107,0,107,12,0,0,0,21.17900,"[2808, 2877, 1566, 1361, 1262, 1018, 750, 9414...",6436,0.008120,-117.511757,2.996826,2.971953,0.114534,4003.046875
108,0,108,11,0,0,0,24.39280,"[2871, 3252, 5637, 5962, 6567, 6903, 7237, 149...",2248,-0.153666,-11.456075,3.019091,0.138767,0.116928,-700.848083


In [76]:
build_hdf5_events(full_track_df, "tracks.h5")

In [77]:
event_data = read_event_tracks('tracks.h5', event_id=0)

In [80]:
event_data["tracks"]["chi2"]

array([19.2416 , 16.7654 , 27.0743 ,  9.89781, 18.0702 , 25.445  ,
       20.9111 , 31.5419 , 16.506  , 38.2656 , 15.2142 , 30.6871 ,
       24.5749 , 48.983  , 30.4101 , 20.1971 , 20.6532 , 33.5792 ,
       36.0777 , 34.0572 , 27.2528 , 15.838  , 36.8548 , 17.301  ,
       22.0968 , 17.3892 , 13.5267 , 17.1586 , 31.5536 , 19.1668 ,
       28.0403 , 13.3335 , 29.0532 , 24.214  , 19.6441 , 17.9912 ,
       34.5462 , 24.0366 , 10.7094 , 35.9662 , 38.1884 , 25.1118 ,
       11.6532 , 26.5172 , 14.1078 , 15.4939 , 28.1643 , 25.8352 ,
       17.4874 , 33.2448 , 32.725  , 22.3165 , 16.2609 , 24.5185 ,
       16.9739 , 18.6277 , 32.9911 , 19.9323 , 20.6941 , 14.1369 ,
       14.05   , 34.994  , 25.2582 , 21.1736 , 29.9934 , 17.1702 ,
       39.5667 , 23.9989 , 12.9616 , 25.9103 , 18.1449 , 21.0361 ,
       14.4848 , 20.7249 ,  7.42721, 13.0655 , 19.7859 , 31.7115 ,
       18.8942 , 26.9478 , 26.9665 , 23.1767 , 31.6454 , 19.596  ,
       21.4015 , 15.8592 , 19.3461 , 26.4139 , 18.084  , 25.21

## Process all events

In [ ]:
def get_majority_particle_id(track_id, tracks_df, simhits_root_df, particle_barcode_map):
    """Calculate the majority particle ID for a given track.
    
    Args:
        track_id: ID of the track to analyze
        tracks_df: DataFrame containing track information
        simhits_root_df: DataFrame containing simulated hits
        particle_barcode_map: Mapping between particle IDs and barcodes
        
    Returns:
        The most common particle barcode for hits in this track
    """
    # Get the hits for this track
    track_hits = simhits_root_df.iloc[tracks_df.Hits_ID.apply(convert_hit_ids)[track_id]]
    
    # Map particle IDs to barcodes and get the most common one
    return track_hits.particle_id.map(particle_barcode_map).mode()[0]

# Apply to all tracks
majority_particle_ids = tracks_df.index.map(lambda x: get_majority_particle_id(x, tracks_df, simhits_root_df, particle_barcode_map))


In [154]:
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import h5py
from typing import Dict, List, Any
from tqdm import tqdm

def get_run_paths(base_dir: str) -> List[str]:
    """
    Get all run directories in the dataset, properly sorted.
    """
    # Get all run directories and sort numerically by run number
    run_dirs = glob.glob(f"{base_dir}/runs/*/")
    return sorted(run_dirs, key=lambda x: int(x.rstrip('/').split('/')[-1]))

def process_event_for_tracks(
    run_dir: str,
    local_event_num: int,
    global_event_num: int,
    tracksummary_arrays: Any,
    tracks_csv_pattern: str,
    simhits_df: pd.DataFrame,
    edm4hep_hits_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Process a single event.
    
    Args:
        run_dir: Path to run directory
        local_event_num: Event number within the run
        global_event_num: Global event number across all runs
        tracksummary_arrays: Arrays from tracksummary ROOT file
        tracks_csv_pattern: Pattern for tracks CSV filenames
        simhits_df: DataFrame containing simhits data
        edm4hep_hits_df: DataFrame containing edm4hep hits data
        
    Returns:
        DataFrame containing track data for this event
    """
    # Load tracks CSV
    tracks_csv = pd.read_csv(f"{run_dir}/{tracks_csv_pattern.format(local_event_num)}")
    
    # Get track summary data for this event
    arrays = tracksummary_arrays[local_event_num]
    track_data = {}
    for field in arrays.fields:
        if field == 'event_nr':
            continue
        try:
            array = ak.to_numpy(arrays[field])
            assert len(array.shape) == 1
            track_data[field] = array
        except Exception:
            continue
            
    track_fitting_df = pd.DataFrame(track_data).rename(columns={"track_nr": "track_id"})

    # Build particle ID - particle barcode mapping
    particle_barcode_map = create_particle_barcode_map(edm4hep_hits_df, simhits_df)

    # Get majority particle ID for each track
    majority_particle_ids = tracks_csv.track_id.map(lambda x: get_majority_particle_id(x, tracks_csv, simhits_df, particle_barcode_map))    
    
    # Combine data
    track_finding_data = {
        "event_id": global_event_num,
        "track_id": tracks_csv.track_id.values,
        "num_hits": tracks_csv.nMeasurements.values,
        "num_outliers": tracks_csv.nOutliers.values,
        "num_holes": tracks_csv.nHoles.values,
        "num_shared_hits": tracks_csv.nSharedHits.values,
        "chi2": tracks_csv.chi2.values,
        "hit_ids": tracks_csv.Hits_ID.apply(convert_hit_ids).values,
        "majority_particle_id": majority_particle_ids.values,
    }
    
    track_fitting_data = {
        "event_id": global_event_num,
        "track_id": track_fitting_df.track_id.values,
        "d0": track_fitting_df.eLOC0_fit.values,
        "z0": track_fitting_df.eLOC1_fit.values,
        "phi": track_fitting_df.ePHI_fit.values,
        "theta": track_fitting_df.eTHETA_fit.values,
        "qop": track_fitting_df.eQOP_fit.values,
        "time": track_fitting_df.eT_fit.values,
    }
    
    full_track_df = pd.DataFrame(track_finding_data)
    event_df = full_track_df.merge(pd.DataFrame(track_fitting_data), 
                                 on=["event_id", "track_id"])
    return event_df

def process_run_for_tracks(
    run_dir: str,
    run_number: int,
    run_size: int = 10,
    tracks_csv_pattern: str = "event{:09d}-tracks_ambi.csv",
    tracksummary_file: str = "tracksummary_ambi.root",
    simhits_file: str = "simhits.root",
    edm4hep_file: str = "edm4hep.root"
) -> List[pd.DataFrame]:
    """
    Process all events in a single run.
    
    Args:
        run_dir: Path to run directory
        run_number: Run number (for global event numbering)
        run_size: Number of events in each run
        
    Returns:
        List of DataFrames, one for each event in the run
    """
    print(f"Processing run {run_number} at {run_dir}")
    
    # Open tracksummary file once for the whole run
    tracksummary_root = uproot.open(f"{run_dir}/{tracksummary_file}")["tracksummary"]
    tracksummary_arrays = tracksummary_root.arrays()

    # Open the simhits root file once for the whole run
    simhits_df = load_root_file(f"{run_dir}/{simhits_file}")
    edm4hep_events_root = uproot.open(f"{run_dir}/{edm4hep_file}")["events"]
    edm4hep_hits_df = get_particle_ids_from_events(edm4hep_events_root, all_tracker_readouts)
    
    run_events = []
    for local_event_num in range(run_size):
        try:
            # Calculate global event number
            global_event_num = run_number * run_size + local_event_num
            
            event_df = process_event_for_tracks(
                run_dir,
                local_event_num,
                global_event_num,
                tracksummary_arrays,
                tracks_csv_pattern,
                simhits_df,
                edm4hep_hits_df
            )
            run_events.append(event_df)
            
        except FileNotFoundError:
            print(f"Skipping missing event {local_event_num} in {run_dir}")
            continue
            
    return run_events

def process_chunk_for_tracks(
    run_dirs: List[str],
    start_run: int,
    num_runs: int,
    output_dir: str,
    dataset_name: str,
    run_size: int = 10
) -> pd.DataFrame:
    """
    Process a chunk of runs and save to HDF5.
    """
    all_events_data = []
    
    # Process each run in the chunk
    for run_idx in range(start_run, min(start_run + num_runs, len(run_dirs))):
        run_dir = run_dirs[run_idx]
        run_events = process_run_for_tracks(run_dir, run_idx, run_size)
        all_events_data.extend(run_events)
    
    if all_events_data:
        combined_df = pd.concat(all_events_data, ignore_index=True)
        
        # Calculate event range for filename
        start_event = start_run * run_size
        end_event = min((start_run + num_runs) * run_size - 1, 
                       len(run_dirs) * run_size - 1)
        
        output_file = f"{output_dir}/{dataset_name}.events{start_event}-{end_event}.h5"
        build_hdf5_tracks(combined_df, output_file)
        print(f"Saved {output_file}")
        
        return combined_df

def process_full_dataset_for_tracks(
    base_dir: str,
    output_base_dir: str,
    chunk_size: int = 1000,
    run_size: int = 10,
    dataset_name: str = "pileup-10/ttbar/v1/reco/tracks/"
) -> None:
    """
    Process entire dataset in chunks.
    
    Args:
        base_dir: Base directory containing all runs
        output_dir: Directory to save HDF5 files
        chunk_size: Number of events per chunk/file
        dataset_name: Name of the dataset (used in filename)
    """
    # Get properly sorted run directories
    run_dirs = get_run_paths(base_dir)
    num_runs = len(run_dirs)
    num_events = num_runs * run_size
    
    # Calculate runs per chunk to achieve desired events per chunk
    runs_per_chunk = chunk_size // run_size
    
    print(f"Processing {num_runs} runs with {num_events} total events")
    print(f"Processing {runs_per_chunk} runs per chunk to get ~{chunk_size} events per file")
    
    output_dir = f"{output_base_dir}/{dataset_name}"
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    dataset_name = dataset_name.replace("/", ".")
    
    # Process chunks of runs
    for start_run in tqdm(range(0, num_runs, runs_per_chunk), desc="Processing chunks"):
        process_chunk_for_tracks(
            run_dirs, 
            start_run, 
            runs_per_chunk, 
            output_dir, 
            dataset_name,
            run_size
        )



In [178]:
run_events = process_run_for_tracks("/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0", 0, 10)

Processing run 0 at /eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0


In [182]:
run_events[0]

,event_id,track_id,num_hits,num_outliers,num_holes,num_shared_hits,chi2,hit_ids,majority_particle_id,d0,z0,phi,theta,qop,time
0,0,109,11,0,0,0,22.2287,"[2831, 2911, 3260, 4286, 4298, 5133, 5145, 103...",6662,-0.009959,-105.365746,2.993862,0.454155,0.422710,2435.543701
1,0,108,11,0,0,0,24.3928,"[2871, 3252, 5637, 5962, 6567, 6903, 7237, 149...",2907,-0.153666,-11.456075,3.019091,0.138767,0.116928,-700.848083
2,0,107,12,0,0,0,21.1790,"[2808, 2877, 1566, 1361, 1262, 1018, 750, 9414...",1730,0.008120,-117.511757,2.996826,2.971953,0.114534,4003.046875
3,0,106,13,0,0,0,13.8609,"[2914, 3290, 4295, 4312, 5153, 5160, 10397, 11...",322277,-0.044598,-116.748528,3.075047,0.570654,-0.268563,4014.498535
4,0,105,17,0,0,0,23.8647,"[2912, 3257, 3287, 4294, 5152, 10420, 10354, 1...",2093,-0.020431,-41.542458,3.001691,1.752115,-0.820906,341.259796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,0,54,14,0,0,0,16.9739,"[2390, 1829, 3820, 4754, 5554, 5563, 11096, 12...",6154,-0.036438,-11.799022,-0.245146,0.734197,-0.557603,-712.012085
106,0,55,11,0,0,0,18.6277,"[2415, 5739, 6071, 6437, 6983, 7314, 7644, 797...",1376,0.081550,-105.254639,-0.282000,0.105319,-0.014828,2414.812012
107,0,56,12,0,0,0,32.9911,"[1805, 3047, 3085, 4032, 4842, 4954, 9992, 113...",1378,-0.044134,-105.720299,0.104896,2.389486,-0.225617,2418.907227
108,0,57,14,0,0,0,19.9323,"[1813, 1904, 3095, 4222, 5068, 10387, 11729, 1...",566,0.030983,-41.453045,0.172066,1.625189,-0.998300,342.496948


In [ ]:
base_dir = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1"
output_base_dir = "/eos/home-d/dmurnane/ColliderML/staging/"
dataset_name = "pileup-10/ttbar/v1/reco/tracks"
last_combined_df = process_full_dataset_for_tracks(base_dir, output_base_dir, dataset_name=dataset_name)

Processing 128 runs with 1280 total events
Processing 100 runs per chunk to get ~1000 events per file


Processing chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Processing run 0 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/0/
Processing run 1 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/1/
Processing run 2 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/2/
Processing run 3 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/3/
Processing run 4 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/4/
Processing run 5 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/5/
Processing run 6 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/6/
Processing run 7 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/7/
Processing run 8 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/8/
Processing run 9 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/9/
Processing run 10 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/10/
Processing run 11 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/11/


Processing chunks:  50%|█████     | 1/2 [00:34<00:34, 34.63s/it]

Saved /eos/home-d/dmurnane/ColliderML/staging//pileup-10/ttbar/v1/reco/tracks//pileup-10.ttbar.v1.reco.tracks..events0-999.h5
Processing run 100 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/100/
Processing run 101 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/101/
Processing run 102 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/102/
Processing run 103 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/103/
Processing run 104 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/104/
Processing run 105 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/105/
Processing run 106 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/106/
Processing run 107 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/107/
Processing run 108 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/108/
Processing run 109 at /eos/home-d/dmurnane/ColliderML/simulation/gg2ttbar/v1/runs/109/
Proc

Processing chunks: 100%|██████████| 2/2 [00:44<00:00, 22.07s/it]

Saved /eos/home-d/dmurnane/ColliderML/staging//pileup-10/ttbar/v1/reco/tracks//pileup-10.ttbar.v1.reco.tracks..events1000-1279.h5


In [130]:
def read_event_tracks(
    input_file: str,
    event_id: int
) -> pd.DataFrame:
    """
    Read a single event from an HDF5 file.
    
    Args:
        input_file: Path to HDF5 file
        event_id: Event ID to read
        
    Returns:
        DataFrame containing track data for the event
    """
    with h5py.File(input_file, 'r') as f:
        event_group = f[f'events/event_{event_id}']
        
        # Read track data and hit arrays
        tracks = event_group['tracks'][:]
        hit_ids = event_group['hit_ids'][:]
        
        # Convert structured array to DataFrame
        df = pd.DataFrame(tracks)
        df['hit_ids'] = hit_ids
        df['event_id'] = event_id
        return df

def read_chunk_tracks(
    input_file: str
) -> pd.DataFrame:
    """
    Read all events from an HDF5 file.
    
    Args:
        input_file: Path to HDF5 file
        
    Returns:
        DataFrame containing track data for all events
    """
    events_data = []
    
    with h5py.File(input_file, 'r') as f:
        events_group = f['events']
        
        # Read each event
        for event_name in events_group:
            event_id = int(event_name.split('_')[1])
            event_df = read_event_tracks(input_file, event_id)
            events_data.append(event_df)
    
    return pd.concat(events_data, ignore_index=True)

def read_events_tracks(
    base_dir: str,
    event_ids: List[int],
    dataset_name: str = "mu10.ttbar"
) -> pd.DataFrame:
    """
    Read specific events from HDF5 files.
    
    Args:
        base_dir: Directory containing HDF5 files
        event_ids: List of event IDs to read
        dataset_name: Name of the dataset
        
    Returns:
        DataFrame containing track data for requested events
    """
    events_data = []
    chunk_size = 1000  # Must match the chunk size used in writing
    
    for event_id in event_ids:
        # Calculate which file contains this event
        chunk_num = event_id // chunk_size
        start_event = chunk_num * chunk_size
        
        # Find the file starting with this event using glob
        import glob
        pattern = f"{base_dir}/{dataset_name}.tracks.events{start_event}-*.h5"
        matching_files = glob.glob(pattern)
        
        if not matching_files:
            print(f"Could not find file containing event {event_id}")
            continue
            
        # Use the first (and should be only) matching file
        filename = matching_files[0]
        
        try:
            event_df = read_event_tracks(filename, event_id)
            events_data.append(event_df)
        except KeyError as e:
            print(f"Could not read event {event_id}: {e}")
            continue
    
    return pd.concat(events_data, ignore_index=True) if events_data else pd.DataFrame()

In [132]:
# Read a single event
event_df = read_event_tracks("/eos/home-d/dmurnane/ColliderML/staging/pileup-10/ttbar/v1/tracks/pileup-10.ttbar.tracks.events1000-1279.h5", event_id=1042)
event_df


,track_id,num_hits,num_outliers,num_holes,num_shared_hits,chi2,d0,z0,phi,theta,qop,time,hit_ids,event_id
0,164,13,0,0,0,23.1875,-0.039698,-5.298965,3.121400,0.320957,0.180174,735.098999,"[3862, 4319, 5603, 6666, 12550, 14204, 17133, ...",1042
1,163,13,0,1,0,11.4459,0.066013,27.154177,2.963757,1.061859,0.815005,90.850555,"[3741, 3844, 4278, 5579, 6648, 12419, 14168, 1...",1042
2,162,14,0,0,0,13.3326,0.045147,27.164377,2.866811,1.128076,-0.696292,120.285149,"[3742, 4276, 5578, 6649, 12465, 14170, 15398, ...",1042
3,161,7,1,1,0,25.1517,0.500882,-5.190934,2.903472,0.318327,0.116799,742.172119,"[3761, 3850, 4280, 5562, 12431, 14177, 17125, ...",1042
4,160,11,0,0,0,12.1697,-0.153048,-12.402186,2.713961,2.983665,-0.119664,122.060089,"[3724, 4211, 1992, 1886, 1563, 1220, 11434, 11...",1042
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,122,11,0,0,0,23.8583,-0.076850,-11.756917,1.200101,0.233436,-0.178868,136.682999,"[3328, 5083, 5289, 7510, 13611, 17088, 17082, ...",1042
161,123,10,0,0,0,29.6108,0.039063,-6.575345,1.349514,0.332730,0.229714,749.103943,"[3318, 3454, 5165, 5304, 6369, 13606, 13774, 1...",1042
162,124,14,0,0,0,22.2595,-0.042631,-6.400096,1.331778,0.488500,0.085002,756.794434,"[3310, 3428, 5158, 5316, 6363, 6378, 13599, 13...",1042
163,125,16,0,0,0,15.8998,-0.054486,-6.147042,1.338427,0.344015,0.317251,753.168945,"[3320, 3456, 5163, 5305, 6368, 13607, 13770, 1...",1042


In [133]:
# Read all events from a file
chunk_df = read_chunk_tracks("/eos/home-d/dmurnane/ColliderML/staging/pileup-10/ttbar/v1/tracks/pileup-10.ttbar.tracks.events1000-1279.h5")
chunk_df


,track_id,num_hits,num_outliers,num_holes,num_shared_hits,chi2,d0,z0,phi,theta,qop,time,hit_ids,event_id
0,95,13,0,0,0,21.4112,-1.430283,-75.022644,2.981686,0.364788,0.327181,-746.019531,"[3162, 4050, 4733, 8967, 8953, 10040, 12187, 1...",1000
1,94,11,0,0,0,14.0715,0.075230,-75.330460,3.083613,0.257651,-0.157941,-745.548340,"[2869, 3182, 4074, 9019, 12094, 12434, 12714, ...",1000
2,93,13,0,0,0,15.9177,-0.016234,26.583233,3.002800,1.786184,-0.901835,-1022.789490,"[2868, 3161, 3173, 4067, 4749, 9008, 10065, 11...",1000
3,92,16,0,0,0,28.4072,0.003843,-30.124830,2.582220,1.993586,0.815352,-2041.186279,"[2731, 2796, 3113, 4019, 4708, 4722, 8927, 100...",1000
4,91,10,0,0,0,17.5300,-0.041078,-30.576918,2.234975,0.114416,-0.065777,-2050.970703,"[2685, 5342, 5591, 5852, 6100, 6346, 6575, 128...",1000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25628,54,16,0,0,0,22.8926,0.054685,-15.385065,0.240472,3.011516,-0.067107,-855.041687,"[2481, 2602, 2132, 1783, 1441, 1697, 1655, 134...",1279
25629,55,13,1,0,0,41.7644,0.013610,58.987488,0.563066,0.113744,-0.097240,-1727.449829,"[2570, 3174, 7661, 7966, 8274, 8595, 8800, 918...",1279
25630,56,11,0,0,0,21.3167,-0.026316,111.258202,0.695506,2.960742,-0.024182,789.598022,"[3228, 4944, 5075, 2418, 2084, 12612, 12196, 1...",1279
25631,57,12,0,0,0,32.0775,-0.034805,111.586349,0.900203,3.023534,0.032647,764.993652,"[3214, 5050, 5107, 2265, 1921, 1754, 1418, 105...",1279


In [134]:
# Read specific events
events_df = read_events_tracks(
    "/eos/home-d/dmurnane/ColliderML/staging/pileup-10/ttbar/v1/tracks",
    event_ids=[0, 42, 999, 1010],
    dataset_name="pileup-10.ttbar"
)
events_df

,track_id,num_hits,num_outliers,num_holes,num_shared_hits,chi2,d0,z0,phi,theta,qop,time,hit_ids,event_id
0,109,11,0,0,0,22.2287,-0.009959,-105.365746,2.993862,0.454155,0.422710,2435.543701,"[2831, 2911, 3260, 4286, 4298, 5133, 5145, 103...",0
1,108,11,0,0,0,24.3928,-0.153666,-11.456075,3.019091,0.138767,0.116928,-700.848083,"[2871, 3252, 5637, 5962, 6567, 6903, 7237, 149...",0
2,107,12,0,0,0,21.1790,0.008120,-117.511757,2.996826,2.971953,0.114534,4003.046875,"[2808, 2877, 1566, 1361, 1262, 1018, 750, 9414...",0
3,106,13,0,0,0,13.8609,-0.044598,-116.748528,3.075047,0.570654,-0.268563,4014.498535,"[2914, 3290, 4295, 4312, 5153, 5160, 10397, 11...",0
4,105,17,0,0,0,23.8647,-0.020431,-41.542458,3.001691,1.752115,-0.820906,341.259796,"[2912, 3257, 3287, 4294, 5152, 10420, 10354, 1...",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,54,13,0,0,0,13.7770,-0.019463,-62.298725,0.496364,1.060234,-0.202174,-833.581238,"[1356, 2317, 2528, 3092, 3604, 6643, 7472, 809...",1010
337,55,14,0,0,0,24.9239,0.160899,-79.498787,0.472725,2.957299,-0.073057,843.923523,"[1344, 2312, 2514, 1167, 1091, 910, 741, 5813,...",1010
338,56,15,0,0,0,19.9348,0.017474,-62.305973,0.667609,0.860790,0.098933,-850.834045,"[1731, 2522, 2589, 3182, 3205, 3709, 6825, 753...",1010
339,57,17,0,0,0,19.4698,0.010124,-62.297741,0.636527,0.888784,0.135478,-830.512512,"[1361, 1714, 2527, 3179, 3712, 6824, 7534, 747...",1010
